# Phase 00 ? ki?m tra Colab

Ch?y l?n l??t c?c cell. D?ng kit ZIP ?i k?m repo hi?n t?i; kh?ng train full, kh?ng ghi ?? artifact th?c t?p. Synthetic probes v? real-prefix pilot ch? ki?m tra t??ng th?ch k? thu?t. Lab b? qua theo y?u c?u ng??i d?ng.

In [ ]:
import sys, subprocess
import torch
print("Python:", sys.version)
print("Torch:", torch.__version__, "CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
subprocess.run([sys.executable, "-m", "pip", "install", "torch-geometric==2.8.0", "pandas>=2.2", "pyarrow>=15", "scikit-learn>=1.4", "networkx>=3.2", "joblib>=1.3", "psutil"], check=True)


## Upload code kit
Ch?n colab_phase00_kit.zip t? results/graduation/phase_00/ tr?n m?y. G?i ch? ch?a code; kh?ng c?n upload CSV l?n qua cell n?y.

In [ ]:
from google.colab import files
from pathlib import Path
import tempfile, zipfile
uploaded = files.upload()
kit_name = next(name for name in uploaded if name.endswith(".zip"))
PROJECT_DIR = Path(tempfile.mkdtemp(prefix="gnn_phase00_", dir="/content"))
with zipfile.ZipFile(kit_name) as archive:
    for name in archive.namelist():
        target = (PROJECT_DIR / name).resolve()
        if not target.is_relative_to(PROJECT_DIR.resolve()):
            raise ValueError("Invalid archive path")
    archive.extractall(PROJECT_DIR)
assert (PROJECT_DIR / "scripts/phase00_audit.py").is_file()
print("Code:", PROJECT_DIR)


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DATA_ROOT = Path("/content/drive/MyDrive/GNN-NIDS_TTTN/data")
OUTPUT_ROOT = Path("/content/drive/MyDrive/GNN-NIDS_TTTN/graduation/phase_00")
SCAN_FULL_LABELS = False
print("Data root exists:", DATA_ROOT.is_dir())


In [ ]:
from datetime import datetime, timezone
RUN_DIR = OUTPUT_ROOT / datetime.now(timezone.utc).strftime("colab_%Y%m%dT%H%M%S_%fZ")
command = [sys.executable, "-u", str(PROJECT_DIR / "scripts/phase00_audit.py"), "--data-root", str(DATA_ROOT), "--output-dir", str(RUN_DIR)]
command += ["--scan-raw"] if SCAN_FULL_LABELS else ["--skip-dataset"]
subprocess.run(command, cwd=PROJECT_DIR, check=True)


In [ ]:
import json
result = json.loads((RUN_DIR / "audit.json").read_text(encoding="utf-8"))
print("Saved:", RUN_DIR)
for name, section in result["sections"].items():
    print(name, section["status"], round(section["seconds"], 3), "seconds")
print(json.dumps(result["sections"].get("real_multiclass_pilot"), indent=2))


## K?t lu?n ??ng ph?m vi
GPU Colab ch?a ???c x?c nh?n cho t?i khi cell th?c ch?y. Pilot 64-flow kh?ng ??i di?n c?a s? 2.000/500 flow, kh?ng c? held-out test v? kh?ng ch?ng minh ?? ch?nh x?c. K?t qu? JSON c?n ???c ??i chi?u tr??c khi tick nghi?m thu P0.